# Reproduce the spectra of the mechanical-mode-comb paper
Reproduces with levisim the heterodyne and homodyne spectra of J. H. Iacoponi, M. Rademacher and T. S. Monteiro, *Nanoparticle arrays levitated in a cavity for quantum sensing*, Phys. Rev. Research **8** (2026), [arXiv:2406.00366](https://arxiv.org/abs/2406.00366) (first posted as *Collective modes of nanoparticles levitated in a cavity: mechanical mode combs*). The figure numbers in this notebook follow arXiv v1, which it was written against; the published version (arXiv v3) dropped two of those figures and added a comparison with its closed-form Eq. (17):

| this notebook | arXiv v1 | published (PRR 8) |
|---|---|---|
| Fig 1(b): heterodyne, 16 modes, equal spacing / pairs / random | Fig 1(b) | Fig 1(b) |
| Fig 3(b): heterodyne, 1 and 5 particles, comb spacing δ = 1, 4, 15 kHz | Fig 3(b) | - |
| Fig 4: homodyne (ponderomotive squeezing), 4, 8, 16 modes at 10⁻⁸ mbar | Fig 4 | - |
| Fig 5: heterodyne, 2 particles in 2D, detuning sweep | Fig 5 | Fig 5 |
| PRR Fig 4: Eq. (17) vs. numerics, 4 particles, equal and unequal tweezer phases | - | Fig 4 |

The spectra are computed with the linear amplifier model (LAM). Where the paper overlays the closed form (Fig 1(b) and PRR Fig 4), the analytic method (Analytic Output, Eq. (17)) is drawn over LAM and the largest relative difference between the two is printed. The particle-loss figure (arXiv v1 Fig 2, PRR Fig 3) and the CBM/MMC avoided-crossing maps (arXiv v1 Fig 3(a), PRR Fig 2) are not reproduced here.

Runtime: about 1 min.

# START

# Reproduce Figs 1, 3, 4, 5 of MMC paper

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from levisim import *

set_plt_rcParams_to_xmgrace_style()

# which figures to draw
Fig_1 = True
Fig_3 = True
Fig_4 = True
Fig_5 = True
PRR_Fig_4 = True

LAM = QuantumLinearTheoryMethod.LINEAR_AMPLIFIER_MODEL
ANALYTIC = QuantumLinearTheoryMethod.ANALYTIC_CAVITY_OUTPUT  # Eq. (17) of the published paper


def make_mtrap(system, ω_range, gas_kwargs, cavity_kwargs, toy_params):
    """A MultiTrap for a system string such as '8×2' (traps × Cartesian axes per trap)"""
    N, d = (int(_) for _ in system.split('×'))
    return MultiTrap(
        num_traps=N,
        Cartesian_axes_per_trap=d,
        ω_range=ω_range,
        gas_kwargs=gas_kwargs,
        cavity_kwargs=cavity_kwargs,
        toy_params=toy_params,
    )


def optical_spectrum(mtrap, spectrum_type, method=LAM):
    """The homodyne or heterodyne spectrum of the cavity output [photons/Hz]"""
    match spectrum_type:
        case SpectrumType.HOMODYNE:
            return mtrap.homodyne_spectrum(method)
        case SpectrumType.HETERODYNE:
            return mtrap.heterodyne_spectrum(method)


def plot_panel(ax, mtrap, spectrum_type, text='', ylims=(1e-1, 1e2), yscale='log', overlay_analytic=False):
    """
    One panel in the style of the paper: the LAM spectrum as a hatched fill, the unperturbed
    frequencies Ω_j as red lines, and (optionally) the analytic closed form drawn over it.
    Returns the largest relative difference between the two methods (or None).
    """
    ω = mtrap.ω_range / kHz
    vals = optical_spectrum(mtrap, spectrum_type, LAM)
    ax.vlines([dim.Ω / kHz for dim in mtrap.dimensions.values()], *ylims, color=red, alpha=0.3, lw=0.8)
    ax.axhline(1, color=black, alpha=0.1)
    ax.fill_between(ω, vals, ylims[0], hatch='.....', edgecolor=purple, facecolor='none', linewidth=0, alpha=0.5)
    max_relative_difference = None
    if overlay_analytic:
        analytic_vals = optical_spectrum(mtrap, spectrum_type, ANALYTIC)
        max_relative_difference = np.max(np.abs(analytic_vals - vals) / np.abs(vals))
        ax.plot(ω, analytic_vals, color=black, lw=1, label='Analytical')
        ax.plot(ω, vals, color=RGB_green, lw=1, ls=':', label='Numerics (LAM)')
    else:
        ax.plot(ω, vals, color=purple, lw=1)
    ax.set_yscale(yscale)
    ax.set_xlim(ω.min(), ω.max())
    # stop just short of the top so that the top tick label does not collide with the panel above
    ax.set_ylim(ylims[0], 0.999 * ylims[1] if yscale == 'log' else ylims[1])
    ax.text(0.03, 0.9, text, transform=ax.transAxes, ha='left', va='top', color=purple, fontsize=10)
    return max_relative_difference


def stacked_axes(num_rows, num_cols=1, height_per_row=1.6, width=LATEX_FIG_WIDTH):
    """A grid of panels with shared axes and no vertical gaps, as in the paper's figures"""
    fig, axes = plt.subplots(
        num_rows, num_cols, figsize=(width, height_per_row * num_rows + 0.8), dpi=100,
        sharex=True, sharey=True, squeeze=False, gridspec_kw=dict(hspace=0, wspace=0.12),
    )
    return fig, axes


## Fig 1(b): the collective bright mode and the mechanical mode comb

In [ ]:
if Fig_1:
    # N_tot = 16 modes (8 particles, x and y), g/2π = 40 kHz, comb span Δω/2π = 100 kHz,
    # -Δ = 1.5 ω_m with ω_m/2π = 300 kHz, κ/2π = 200 kHz (ω_m ≃ 1.5 κ), 10⁻⁷ mbar
    system = '8×2'
    ω_range = np.arange(0, 400 + 0.1, 0.1) * kHz
    gas_kwargs = dict(P=1e-7*mbar, T=300*K, vth=500*metre/second)
    cavity_kwargs = dict(Δ=-450*kHz, κ=200*kHz, θhom=π/4)
    couplings = dict(toy_gx=-40*kHz, toy_gy=-40*kHz)

    # (iii) the paper does not record its random draw, so this panel is a seeded stand-in:
    # 16 frequencies drawn uniformly over the same 250-350 kHz span
    rng = np.random.default_rng(2024)
    labels = [f'{axis}{n}' for n in range(1, 9) for axis in 'xy']
    random_Ωs = np.sort(rng.uniform(250, 350, len(labels)))[::-1] * kHz

    panels = {
        '(i) Equal spacing': dict(Ω_model='equal spacing', Ω_mid=300*kHz, Ω_span=100*kHz),
        '(ii) Pairs': dict(Ω_model='pairs', Ω_first=350*kHz, Ω_intrapair_gap=1.5*kHz, Ω_interpair_gap=15*kHz),
        '(iii) Random': dict(Ω_model='custom', toy_Ωjn_dict=dict(zip(labels, random_Ωs))),
    }
    fig, axes = stacked_axes(len(panels))
    for ax, (text, Ω_model) in zip(axes[:, 0], panels.items()):
        mtrap = make_mtrap(system, ω_range, gas_kwargs, cavity_kwargs, dict(**Ω_model, **couplings))
        difference = plot_panel(ax, mtrap, SpectrumType.HETERODYNE, text, ylims=(0.5, 2e3), overlay_analytic=True)
        print(f'Fig 1(b) {text}: largest relative difference, analytic vs. LAM = {difference:.1e}')
    axes[0, 0].legend(loc='upper right', prop=dict(size=8))
    axes[0, 0].set_title(r'Fig 1(b): $N^\mathrm{tot}$ = 16, heterodyne')
    axes[-1, 0].set_xlabel('Frequency [kHz]')
    fig.supylabel('Heterodyne PSD [photons/Hz]', fontsize=12)
    plt.show()


## Fig 3(b) of arXiv v1: comb spacing

In [ ]:
if Fig_3:
    # N_p = 1 and 5 particles (x and y), equal spacing about 300 kHz with mean spacing δ = 1, 4, 15 kHz,
    # g/2π = 40 kHz, 10⁻⁷ mbar
    ω_range = np.arange(0, 400 + 0.1, 0.1) * kHz
    gas_kwargs = dict(P=1e-7*mbar, T=300*K, vth=500*metre/second)
    cavity_kwargs = dict(Δ=-450*kHz, κ=200*kHz, θhom=π/4)
    systems = ['1×2', '5×2']
    δs = [1, 4, 15]
    fig, axes = stacked_axes(len(δs), len(systems), width=1.6 * LATEX_FIG_WIDTH)
    for col, system in enumerate(systems):
        N, d = (int(_) for _ in system.split('×'))
        for row, δ in enumerate(δs):
            span = δ * (N * d - 1)
            toy_params = dict(Ω_model='equal spacing', Ω_mid=300*kHz, Ω_span=span*kHz, toy_gx=-40*kHz, toy_gy=-40*kHz)
            mtrap = make_mtrap(system, ω_range, gas_kwargs, cavity_kwargs, toy_params)
            plot_panel(axes[row, col], mtrap, SpectrumType.HETERODYNE, rf'δ = {δ} kHz (span = {span} kHz)', ylims=(1e-1, 1e2))
        axes[0, col].set_title(rf'$N_\mathrm{{p}}$ = {N}')
        axes[-1, col].set_xlabel('Frequency [kHz]')
    fig.suptitle('Fig 3(b) of arXiv v1: heterodyne spectra vs. comb spacing')
    fig.supylabel('Heterodyne PSD [photons/Hz]', fontsize=12)
    plt.show()


## Fig 4 of arXiv v1: homodyne spectra and ponderomotive squeezing

In [ ]:
if Fig_4:
    # the pairs of Fig 1(b)(ii), homodyne angle π/4, at 10⁻⁸ mbar so that the quantum shot noise dominates
    ω_range = np.arange(0, 600 + 0.1, 0.1) * kHz
    gas_kwargs = dict(P=1e-8*mbar, T=300*K, vth=500*metre/second)
    cavity_kwargs = dict(Δ=-450*kHz, κ=200*kHz, θhom=π/4)
    toy_params = dict(Ω_model='pairs', Ω_first=350*kHz, Ω_intrapair_gap=1.5*kHz, Ω_interpair_gap=15*kHz, toy_gx=-40*kHz, toy_gy=-40*kHz)
    systems = ['2×2', '4×2', '8×2']
    fig, axes = stacked_axes(len(systems), height_per_row=1.8)
    for ax, system in zip(axes[:, 0], systems):
        mtrap = make_mtrap(system, ω_range, gas_kwargs, cavity_kwargs, toy_params)
        plot_panel(ax, mtrap, SpectrumType.HOMODYNE, rf'$N^\mathrm{{tot}}$ = {mtrap.num_dimensions}', ylims=(0, 5), yscale='linear')
    axes[0, 0].set_title(r'Fig 4 of arXiv v1: ponderomotive squeezing, $P$ = 10$^{-8}$ mbar')
    axes[-1, 0].set_xlabel('Frequency [kHz]')
    fig.supylabel('Homodyne PSD [photons/Hz]', fontsize=12)
    plt.show()


## Fig 5: two particles in 2D, detuning sweep

In [ ]:
if Fig_5:
    # N_p = 2 particles (x and y), 10⁻⁷ mbar: (a) equal spacing, (b) pairs, at four detunings each
    ω_range = np.arange(100, 350 + 0.1, 0.1) * kHz
    gas_kwargs = dict(P=1e-7*mbar, T=300*K, vth=500*metre/second)
    couplings = dict(toy_gx=-40*kHz, toy_gy=-40*kHz)
    columns = {
        '(a) Equal spacing': (dict(Ω_model='equal spacing', Ω_mid=265*kHz, Ω_span=15*kHz), [-2000, -750, -450, -250]),
        # the paper's top right panel is at -4000 kHz (the owner's earlier version of this notebook used -2000)
        '(b) Pairs': (dict(Ω_model='pairs', Ω_first=265*kHz, Ω_intrapair_gap=1.5*kHz, Ω_interpair_gap=15*kHz), [-4000, -750, -450, -250]),
    }
    fig, axes = stacked_axes(4, 2, width=1.6 * LATEX_FIG_WIDTH)
    for col, (title, (Ω_model, detunings)) in enumerate(columns.items()):
        for row, detuning in enumerate(detunings):
            cavity_kwargs = dict(Δ=detuning*kHz, κ=200*kHz, θhom=π/4)
            mtrap = make_mtrap('2×2', ω_range, gas_kwargs, cavity_kwargs, dict(**Ω_model, **couplings))
            plot_panel(axes[row, col], mtrap, SpectrumType.HETERODYNE, rf'$\Delta/2\pi$ = {detuning} kHz', ylims=(5e-1, 1e1))
        axes[0, col].set_title(title)
        axes[-1, col].set_xlabel('Frequency [kHz]')
    fig.suptitle(r'Fig 5: 2 nanoparticle heterodyne spectra ($N_\mathrm{p}$ = 2)')
    fig.supylabel('Heterodyne PSD [photons/Hz]', fontsize=12)
    plt.show()


## Fig 4 of the published version (PRR 8): Eq. (17) vs. numerics with tweezer phases
The published paper does not state the unequal phases in the text; its figure labels them Φ₁ = 0°, Φ₂ = 30°, Φ₃ = 60°, Φ₄ = 90°, which is what is used here.

In [ ]:
if PRR_Fig_4:
    # N = 4 particles, one axis each: Ω_j/2π = 300, 301.5, 285, 286.5 kHz, g_j/2π = 40, 38, 36, 34 kHz,
    # Δ/2π = -450 kHz, κ/2π = 390 kHz, 10⁻⁶ mbar; all tweezers in phase (upper) and φ_n = 0°, 30°, 60°, 90° (lower)
    ω_range = np.arange(150, 350 + 0.05, 0.05) * kHz
    gas_kwargs = dict(P=1e-6*mbar, T=300*K, vth=500*metre/second)
    cavity_kwargs = dict(Δ=-450*kHz, κ=390*kHz, θhom=π/4)
    base_params = dict(
        Ω_model='pairs', Ω_first=300*kHz, Ω_intrapair_gap=1.5*kHz, Ω_interpair_gap=15*kHz,
        gjn_dict=dict(x1=-40*kHz, x2=-38*kHz, x3=-36*kHz, x4=-34*kHz),
    )
    phase_sets = {
        'equal phases': {1: 0*degrees, 2: 0*degrees, 3: 0*degrees, 4: 0*degrees},
        'different phases': {1: 0*degrees, 2: 30*degrees, 3: 60*degrees, 4: 90*degrees},
    }
    fig, axes = stacked_axes(len(phase_sets), height_per_row=2.4)
    for ax, (text, φn_dict) in zip(axes[:, 0], phase_sets.items()):
        mtrap = make_mtrap('4×1', ω_range, gas_kwargs, cavity_kwargs, dict(**base_params, φn_dict=φn_dict))
        difference = plot_panel(ax, mtrap, SpectrumType.HETERODYNE, phase_per_tweezer_string(mtrap, latex=True, uppercase=True), ylims=(1, 3e3), overlay_analytic=True)
        print(f'PRR 8 Fig 4, {text}: largest relative difference, analytic vs. LAM = {difference:.1e}')
    axes[0, 0].legend(loc='upper right', prop=dict(size=8))
    axes[0, 0].set_title('PRR 8 Fig 4: Eq. (17) vs. numerics, N = 4')
    axes[-1, 0].set_xlabel('Frequency [kHz]')
    fig.supylabel('Heterodyne PSD [photons/Hz]', fontsize=12)
    plt.show()


# END